# Invoice OCR — Gemma 4 31B bản gốc BF16
Linux GPU / Python >= 3.12. Notebook lấy script từ GitHub; **không dùng Ollama, không lượng tử hóa**.
Cần tối thiểu 80 GiB VRAM trống (Molab RTX PRO 6000 thường có 96 GB VRAM). RAM 32 GiB là bộ nhớ hệ thống.


In [ ]:
REPO = "ndkhanhus27/invoice-ocr"
REF = "main"
GITHUB_PRIVATE = False
CONFIG = {
    "github_repo": REPO, "github_ref": REF,
    "dataset": "ndkhanhus27/invoice-ocr-data", "dataset_ref": "main",
    "split": "valid", "limit": 3,  # 0 = toàn bộ reviewed.
    "model": "google/gemma-4-31B-it", "model_ref": "main", "weights_format": "bf16",
    "max_input_tokens": 8192, "max_tokens": 4096, "max_edge": 1600, "vision_soft_tokens": 1120,
}


## Lấy script từ GitHub


In [ ]:
import getpass, importlib.util, json, os, tempfile
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.parse import quote

github_token = os.environ.get("GITHUB_TOKEN", "")
if GITHUB_PRIVATE and not github_token:
    github_token = getpass.getpass("GitHub token (ẩn): ")
headers = {"User-Agent": "invoice-ocr-molab"}
if github_token:
    headers["Authorization"] = "Bearer " + github_token
with urlopen(Request(f"https://api.github.com/repos/{REPO}/commits/{quote(REF, safe='')}", headers=headers), timeout=60) as response:
    commit = json.load(response)["sha"]
with urlopen(Request(f"https://raw.githubusercontent.com/{REPO}/{commit}/scripts/molab_benchmark.py", headers=headers), timeout=60) as response:
    script = Path(tempfile.mkdtemp(prefix="invoice-code-")) / "molab_benchmark.py"
    script.write_bytes(response.read())
spec = importlib.util.spec_from_file_location("molab_benchmark", script)
benchmark = importlib.util.module_from_spec(spec)
spec.loader.exec_module(benchmark)
CONFIG["github_ref"] = commit
print("Đã lấy script GitHub:", commit, flush=True)


## Cài thư viện


In [ ]:
benchmark.install_dependencies(
    requirements_url=f"https://raw.githubusercontent.com/{REPO}/{commit}/requirements/molab.txt",
    github_token=github_token,
)


## Chạy bản gốc và chấm điểm
Cell này kiểm tra GPU, tải full data và weights gốc BF16 từ HF; nạp model một lần, batch 1.
Log/heartbeat hiện trực tiếp. Prediction lưu sau từng hóa đơn tại `invoice-results/<run-id>/results/`.
Các file này vẫn có thể mất nếu Molab thay runtime; tải ZIP hoặc lưu ra remote storage để giữ lâu dài.


In [ ]:
result = benchmark.run_from_notebook(CONFIG, github_token=github_token)


## Điểm và tải ZIP


In [ ]:
benchmark.show_results(result)
